# 04 — Linear Model Search (Speed Prediction)

This notebook demonstrates a **Gaussian linear mixed-effects model search**
for vehicle operating speed relative to raised platforms.

**When to use a linear model (not a count model):**
- Outcome is **continuous and real-valued** (speed, travel time, gap duration)
- Outcome can be negative or positive (e.g. speed deviation)
- Residuals are approximately Gaussian

**Key differences from count models:**
- No exposure offset term (`offset_col=None`)
- Zero-inflation (role 6) is **not applicable** — omit from `default_roles`
- The dispersion parameter is the **residual variance**, not count over-dispersion
- `model='gaussian'` in `fit_manual_model()`

All other features work identically: random parameters, latent classes, correlated randoms,
heterogeneity in means, grouped effects.

**Previous:** [03_cmf_aadt_search.ipynb](03_cmf_aadt_search.ipynb)
**Next:** [05_batch_script_tutorial.ipynb](05_batch_script_tutorial.ipynb)

In [ ]:
import numpy as np
import pandas as pd

from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    SearchOutputConfig,
    load_example_platform_speed_data,
    load_example_platform_gap_duration_data,
    get_help,
)

# Uncomment to read the full linear model guide:
# get_help('linear')

print('Imports OK')

## 1 — Load the platform speed dataset

The `load_example_platform_speed_data()` dataset simulates vehicle speeds
relative to raised pedestrian platforms across 18 different platform types.

In [ ]:
df = load_example_platform_speed_data()

print(f'Rows: {len(df)}  |  Columns: {df.shape[1]}')
print()
print('All columns:')
print(df.columns.tolist())
print()
print('Outcome (SPEED = vehicle speed at platform) summary:')
print(df['SPEED'].describe().round(2))
print()
print('Platform type distribution:')
print(df['PLATFORM_TYPE'].value_counts().sort_index())

In [ ]:
# ── Column description ────────────────────────────────────────────────────────
col_desc = {
    'PLATFORM_ID':      'Observation ID',
    'SPEED':            'Vehicle speed at platform (outcome — km/h or mph)',
    'DIST_TO_PLATFORM': 'Distance from vehicle to platform at measurement',
    'VEHICLE_SPEED':    'Approach speed',
    'RELATIVE_SPEED':   'Speed relative to posted limit',
    'POSTED_SPEED':     'Posted speed limit',
    'APPROACH_ACCEL':   'Deceleration rate approaching platform',
    'PLATFORM_TYPE':    'Platform category (grouping variable)',
    'PLATFORM_HEIGHT':  'Platform height (mm)',
    'PLATFORM_WIDTH':   'Platform width (m)',
    'AT_PLATFORM':      'Binary: 1 = at platform, 0 = approaching',
}
print('Column descriptions:')
for col, desc in col_desc.items():
    if col in df.columns:
        print(f'  {col:<22}: {desc}')

print()
df.head(5)

## 2 — Select variables and set constraints

For linear models:
- **Omit role 6 (ZI)** from `default_roles` — not meaningful for continuous outcomes
- Random parameters still work: capture unobserved driver heterogeneity
- No offset term needed

In [ ]:
# ── Candidate predictor variables ─────────────────────────────────────────────
# Edit this list for your own dataset
variables = [
    'DIST_TO_PLATFORM',   # distance to platform (continuous)
    'POSTED_SPEED',       # posted speed limit
    'APPROACH_ACCEL',     # deceleration rate
    'PLATFORM_HEIGHT',    # physical platform height
    'PLATFORM_WIDTH',     # physical platform width
    'AT_PLATFORM',        # binary: at platform
]

c = (
    ModelConstraints()

    # ZI is not meaningful for continuous speed outcomes
    # (no need for no_zi calls — we omit role 6 from default_roles)

    # Binary indicator: no random effect
    .no_random('AT_PLATFORM')

    # Allow platform dimensions to have random effects
    .allow_random('PLATFORM_HEIGHT', distributions=['normal'])
    .allow_random('PLATFORM_WIDTH',  distributions=['normal'])

    # Deceleration: allow random (lognormal = positive deceleration effect)
    .allow_random('APPROACH_ACCEL', distributions=['normal', 'lognormal'])
)

print('Active constraints:')
print(c)

## 3 — Build the experiment

In [ ]:
builder = ExperimentBuilder(
    df=df,
    id_col='PLATFORM_ID',
    y_col='SPEED',
    offset_col=None,            # no exposure offset for linear models
    group_id_col='PLATFORM_TYPE',  # grouping variable for grouped effects
)

builder.describe()

In [ ]:
# ── What can you change here? ─────────────────────────────────────────────────
#
#   model_family='linear'       : Gaussian likelihood (vs 'count', 'duration', 'cmf')
#
#   default_roles=[0, 1, 2, 3]  : NO role 6 (ZI) for linear models!
#                                  Add 5 for heterogeneity in means
#                                  Add 7, 8 and set max_latent_classes=2 for LC
#
#   R                           : simulation draws (200=fast, 500=stable)

evaluator = builder.build_evaluator(
    variables=variables,
    constraints=c,
    model_family='linear',
    default_roles=[0, 1, 2, 3],   # NO ZI for linear
    max_latent_classes=1,
    mode='single',
    R=200,
)

print('Linear evaluator ready.')
print('Model family: Gaussian (linear)')
print('Variables   :', variables)

## 4 — Run the search

In [ ]:
print('Running linear model search (max_iter=50 — demo) ...')
print()

result = builder.run(
    evaluator=evaluator,
    algo='sa',
    max_iter=50,
    seed=42,
    output_config=SearchOutputConfig(
        output_dir='results',
        experiment_name='linear_speed_search',
        search_description='Gaussian linear mixed-effects search for platform speed',
    ),
)

print('─' * 60)
print(f'  Algorithm      : {result.algorithm}')
print(f'  Best BIC       : {result.best_score:.3f}')
print(f'  Iterations     : {result.iteration}')
print(f'  Time           : {result.elapsed_time:.1f} s')
print(f'  Saved to       : {result.saved_to}')
print('─' * 60)
print()
print('Best linear structure found:')
for k, v in result.best_spec.items():
    if v:
        print(f'  {k:<20}: {v}')

## 5 — Re-fit and inspect the best model

In [ ]:
print('Re-fitting best linear structure with R=500 draws ...')
print()

fit = builder.fit_manual_model(
    manual_spec=result.best_spec,
    model='gaussian',   # ← Gaussian for linear models
    R=500,
)

print('─' * 60)
print('Final linear model fit:')
print('─' * 60)
print(fit)

## 6 — Manual linear specification

Specify the structure explicitly when you have a hypothesis about which variables
should be fixed, random, or correlated.

In [ ]:
manual_spec = builder.make_manual_spec(
    fixed_terms=['DIST_TO_PLATFORM', 'POSTED_SPEED'],
    rdm_terms=['APPROACH_ACCEL:normal'],
    rdm_cor_terms=['PLATFORM_HEIGHT:normal', 'PLATFORM_WIDTH:normal'],
    grouped_terms=[],
    hetro_in_means=['AT_PLATFORM'],
    zi_terms=[],         # always empty for linear models
    membership_terms=[], # empty for single-class model
    dispersion=1,        # residual variance (always 1 for linear)
    latent_classes=1,
)

print('Manual specification:')
for k, v in manual_spec.items():
    print(f'  {k:<20}: {v}')
print()

manual_fit = builder.fit_manual_model(
    manual_spec=manual_spec,
    model='gaussian',
    R=300,
)

print('─' * 60)
print('Manual linear model:')
print('─' * 60)
print(manual_fit)

## 7 — Latent class linear model

If you suspect there are distinct driver sub-groups (e.g. habitual speeders vs
cautious drivers), extend to a 2-class latent class model.

In [ ]:
# Uncomment to run a 2-class LC linear model search:

# c_lc = (
#     ModelConstraints()
#     .no_random('AT_PLATFORM')
#     .allow_random('PLATFORM_HEIGHT', distributions=['normal'])
#     .allow_random('APPROACH_ACCEL',  distributions=['normal'])
# )
#
# evaluator_lc = builder.build_evaluator(
#     variables=variables,
#     constraints=c_lc,
#     model_family='linear',
#     default_roles=[0, 1, 2, 3, 7, 8],  # include membership roles
#     max_latent_classes=2,
#     R=150,
# )
#
# result_lc = builder.run(evaluator_lc, algo='sa', max_iter=50, seed=0)
# print('2-class linear BIC:', result_lc.best_score)
# print('1-class linear BIC:', result.best_score)
# print('ΔBIC (negative = 2-class better):', result_lc.best_score - result.best_score)

print('2-class LC linear search commented out.')
print('Uncomment the block above, or see get_help("latent_class") for more.')

## 8 — Duration bonus: time until another vehicle speeds over the platform

The package also includes a **duration** example for time-to-event data.
Duration models use a lognormal likelihood (positive, right-skewed outcomes).

In [ ]:
# Load the gap-duration dataset
gap_df = load_example_platform_gap_duration_data()

print(f'Gap duration dataset: {len(gap_df)} rows × {gap_df.shape[1]} columns')
print()
print('Outcome (DURATION_UNTIL_NEXT_SPEEDING) summary:')
print(gap_df['DURATION_UNTIL_NEXT_SPEEDING'].describe().round(2))
print()
print('Columns:', gap_df.columns.tolist())
gap_df.head(4)

In [ ]:
# ── Duration search (identical pattern, different model_family and model arg) ─

gap_builder = ExperimentBuilder(
    df=gap_df,
    id_col='PLATFORM_ID',
    y_col='DURATION_UNTIL_NEXT_SPEEDING',
    offset_col=None,    # no offset for duration models
    group_id_col=None,
)

gap_builder.describe()

gap_evaluator = gap_builder.build_evaluator(
    variables=[
        'PRECEDING_VEHICLE_SPEED',
        'FOLLOWING_VEHICLE_SPEED',
        'POSTED_SPEED',
        'PLATFORM_HEIGHT',
        'PLATFORM_WIDTH',
    ],
    model_family='duration',   # ← lognormal likelihood
    default_roles=[0, 1, 2, 3],
    max_latent_classes=1,
    R=200,
)

# Uncomment to run:
# gap_result = gap_builder.run(gap_evaluator, algo='sa', max_iter=50, seed=22)
# gap_fit = gap_builder.fit_manual_model(
#     manual_spec=gap_result.best_spec, model='lognormal', R=500)
# print(gap_fit)

print('Duration evaluator built.  Uncomment the run lines above to execute.')
print()
print('Key difference: model="lognormal" in fit_manual_model()')
print('See get_help("duration") for the full workflow.')

## Model family comparison

| `model_family` | Likelihood | Outcome type | Role 6 (ZI) | `model=` arg |
| --- | --- | --- | --- | --- |
| `'count'` | Poisson / NB | Non-negative integers | Yes | `'nb'` or `'poisson'` |
| `'linear'` | Gaussian | Real-valued continuous | No | `'gaussian'` |
| `'duration'` | Lognormal | Positive, right-skewed | No | `'lognormal'` |
| `'cmf'` | NB with AADT block | Non-negative integers | Yes | `'nb'` |

```python
get_help('linear')    # full linear model guide
get_help('duration')  # full duration model guide
```

**Next:** [05_batch_script_tutorial.ipynb](05_batch_script_tutorial.ipynb)